# First Order Ordinary Differential Equations
# Équations Différentielles Ordinaires du Premier Ordre

**AIMS Master's Programme — ODE Course**

In this notebook we study first-order ODEs of the form $\frac{dy}{dx} = f(x, y)$. We cover separable equations, linear equations, direction fields (champs de directions), and numerical approximation via Euler's method. We conclude with the fundamental existence and uniqueness theorem and an exercise on the SIR model.

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

plt.rcParams.update({'figure.figsize': (8, 5), 'font.size': 12})

## 1. Separable Equations (Équations à variables séparables)

A first-order ODE is **separable** if it can be written as
$$\frac{dy}{dx} = g(x)\,h(y).$$

We separate and integrate: $\int \frac{dy}{h(y)} = \int g(x)\,dx.$

### Example: Exponential population growth

The Malthusian model $\frac{dP}{dt} = rP$ with $P(0) = P_0$ has the exact solution $P(t) = P_0 e^{rt}$. Let us verify this numerically.

In [ ]:
# Separable equation: dP/dt = rP  (population growth / croissance démographique)
r = 0.3   # growth rate
P0 = 10   # initial population

t_span = (0, 10)
t_eval = np.linspace(*t_span, 200)

# Exact solution
P_exact = P0 * np.exp(r * t_eval)

# Numerical solution with solve_ivp
sol = solve_ivp(lambda t, P: r * P, t_span, [P0], t_eval=t_eval)

plt.figure()
plt.plot(t_eval, P_exact, 'b-', label='Exact: $P_0 e^{rt}$', linewidth=2)
plt.plot(sol.t, sol.y[0], 'r--', label='solve_ivp (RK45)', linewidth=2)
plt.xlabel('Time $t$')
plt.ylabel('Population $P(t)$')
plt.title('Exponential Growth: $dP/dt = rP$')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
# Figure: Comparison of exact and numerical solutions for exponential growth.

## 2. Linear First-Order Equations (Équations linéaires du premier ordre)

A **linear** first-order ODE has the form
$$\frac{dy}{dx} + P(x)\,y = Q(x).$$

The integrating factor (facteur intégrant) is $\mu(x) = e^{\int P(x)\,dx}$.

### Example: Mixing problem (problème de mélange)

A 100-litre tank initially contains 10 kg of salt dissolved in water. Brine with concentration 0.5 kg/L flows in at 2 L/min, and the well-stirred mixture flows out at 2 L/min. Let $y(t)$ be the amount of salt (kg) at time $t$.

$$\frac{dy}{dt} = \underbrace{(0.5)(2)}_{\text{in}} - \underbrace{\frac{y}{100}\cdot 2}_{\text{out}} = 1 - \frac{y}{50}$$

This is linear with exact solution $y(t) = 50 - 40\,e^{-t/50}$.

In [ ]:
# Mixing problem: dy/dt = 1 - y/50
y0_mix = 10.0  # initial salt (kg)

t_mix = np.linspace(0, 250, 300)
y_exact_mix = 50 - 40 * np.exp(-t_mix / 50)

sol_mix = solve_ivp(lambda t, y: 1 - y / 50, (0, 250), [y0_mix], t_eval=t_mix)

plt.figure()
plt.plot(t_mix, y_exact_mix, 'b-', label='Exact solution', linewidth=2)
plt.plot(sol_mix.t, sol_mix.y[0], 'r--', label='solve_ivp', linewidth=2)
plt.axhline(y=50, color='gray', linestyle=':', label='Equilibrium $y=50$')
plt.xlabel('Time (min)')
plt.ylabel('Salt (kg)')
plt.title('Mixing Problem (Problème de mélange)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
# Figure: Salt concentration approaches the equilibrium value of 50 kg.

## 3. Direction Fields (Champs de directions)

A **direction field** visualises the slope $f(x,y)$ at each point $(x,y)$. This gives geometric intuition about the solution curves without solving the equation. We use `matplotlib.pyplot.quiver` to draw these fields.

In [ ]:
# Direction field for dy/dx = x - y
x_grid = np.linspace(-3, 3, 20)
y_grid = np.linspace(-3, 3, 20)
X, Y = np.meshgrid(x_grid, y_grid)

# Slope at each point
dY = X - Y
dX = np.ones_like(dY)

# Normalise arrows for uniform length
N = np.sqrt(dX**2 + dY**2)
dX_n, dY_n = dX / N, dY / N

plt.figure()
plt.quiver(X, Y, dX_n, dY_n, color='steelblue', alpha=0.7)

# Overlay a few solution curves via solve_ivp
for y0_val in [-2, 0, 1, 3]:
    sol_dir = solve_ivp(lambda t, y: t - y, (-3, 3), [y0_val],
                        t_eval=np.linspace(-3, 3, 200), max_step=0.05)
    plt.plot(sol_dir.t, sol_dir.y[0], linewidth=1.5)

plt.xlabel('$x$')
plt.ylabel('$y$')
plt.title("Direction field for $dy/dx = x - y$")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
# Figure: Direction field with solution curves for dy/dx = x - y.

## 4. Euler's Method vs `solve_ivp`

**Euler's method** (méthode d'Euler) is the simplest numerical scheme for $y' = f(t, y)$:
$$y_{n+1} = y_n + h\,f(t_n, y_n)$$
where $h$ is the step size (pas). It is first-order accurate: the global error is $O(h)$.

We implement it from scratch and compare with `scipy.integrate.solve_ivp` (which defaults to RK45, a 4th/5th order method).

In [ ]:
def euler_method(f, t_span, y0, h):
    """Forward Euler method (méthode d'Euler explicite).
    
    Parameters
    ----------
    f : callable, f(t, y) -> dy/dt
    t_span : tuple (t0, tf)
    y0 : float, initial value
    h : float, step size
    
    Returns
    -------
    t_arr, y_arr : numpy arrays
    """
    t0, tf = t_span
    t_arr = np.arange(t0, tf + h/2, h)
    y_arr = np.zeros_like(t_arr)
    y_arr[0] = y0
    for i in range(len(t_arr) - 1):
        y_arr[i+1] = y_arr[i] + h * f(t_arr[i], y_arr[i])
    return t_arr, y_arr

# Test on dy/dt = -2y, y(0) = 1  => exact: y = e^{-2t}
f_test = lambda t, y: -2 * y
t_exact = np.linspace(0, 3, 300)
y_exact_test = np.exp(-2 * t_exact)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: solutions
axes[0].plot(t_exact, y_exact_test, 'k-', label='Exact', linewidth=2)
for h_val, color in [(0.5, 'red'), (0.2, 'orange'), (0.05, 'green')]:
    te, ye = euler_method(f_test, (0, 3), 1.0, h_val)
    axes[0].plot(te, ye, 'o-', color=color, markersize=3, label=f'Euler h={h_val}')

sol_ivp = solve_ivp(f_test, (0, 3), [1.0], t_eval=t_exact)
axes[0].plot(sol_ivp.t, sol_ivp.y[0], 'b--', label='solve_ivp (RK45)', linewidth=2)
axes[0].set_xlabel('$t$'); axes[0].set_ylabel('$y$')
axes[0].set_title('Euler vs solve_ivp for $dy/dt = -2y$')
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)

# Right: error at t=3 vs step size
h_values = np.logspace(-3, -0.3, 30)
errors = []
for h_val in h_values:
    te, ye = euler_method(f_test, (0, 3), 1.0, h_val)
    errors.append(abs(ye[-1] - np.exp(-6)))

axes[1].loglog(h_values, errors, 'bo-', markersize=4, label='Euler error')
axes[1].loglog(h_values, h_values, 'r--', label='$O(h)$ reference')
axes[1].set_xlabel('Step size $h$'); axes[1].set_ylabel('|Error at $t=3$|')
axes[1].set_title('Euler method: global error is $O(h)$')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
# Figure: Left — Euler approximations converge to exact solution as h decreases.
# Right — log-log plot confirms first-order convergence.

## 5. Existence and Uniqueness: The Picard-Lindelöf Theorem

**Theorem (Picard-Lindelöf / Cauchy-Lipschitz).** Consider the initial value problem
$$\frac{dy}{dt} = f(t,y), \quad y(t_0) = y_0.$$

If $f$ is **continuous** in a rectangle $R = \{(t,y) : |t - t_0| \le a,\; |y - y_0| \le b\}$ and **Lipschitz continuous in $y$** on $R$, i.e., there exists $L > 0$ such that
$$|f(t, y_1) - f(t, y_2)| \le L|y_1 - y_2| \quad \forall (t, y_1), (t, y_2) \in R,$$
then there exists a **unique** solution $y(t)$ on some interval $|t - t_0| \le \alpha$ where $\alpha = \min(a, b/M)$ and $M = \max_R |f|$.

### Key implications

1. **Solution curves do not cross** in regions where $f$ is Lipschitz.
2. The Lipschitz condition is satisfied whenever $\partial f / \partial y$ exists and is bounded.
3. **Counter-example:** $dy/dt = y^{2/3}$, $y(0)=0$ has $f(t,y) = y^{2/3}$ which is **not** Lipschitz at $y=0$. Indeed, both $y(t) = 0$ and $y(t) = (t/3)^3$ are solutions — uniqueness fails.

### Picard iteration (itération de Picard)

The constructive proof builds successive approximations:
$$y_{n+1}(t) = y_0 + \int_{t_0}^{t} f(s, y_n(s))\,ds$$
converging uniformly to the solution.

In [ ]:
# Illustrating non-uniqueness: dy/dt = y^(2/3), y(0) = 0
# Two solutions: y = 0 and y = (t/3)^3 for t >= 0

t_nu = np.linspace(-1, 3, 300)

# Solution 1: trivial
y_trivial = np.zeros_like(t_nu)

# Solution 2: non-trivial (valid for t >= 0)
y_nontrivial = np.where(t_nu >= 0, (t_nu / 3)**3, 0)

plt.figure()
plt.plot(t_nu, y_trivial, 'b-', linewidth=2, label='$y(t) = 0$')
plt.plot(t_nu, y_nontrivial, 'r-', linewidth=2, label='$y(t) = (t/3)^3$')
plt.plot(0, 0, 'ko', markersize=8, label='$y(0) = 0$ (both pass here)')
plt.xlabel('$t$'); plt.ylabel('$y$')
plt.title('Non-uniqueness when Lipschitz condition fails: $dy/dt = y^{2/3}$')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
# Figure: Two distinct solutions passing through (0,0) — uniqueness fails
# because f(t,y)=y^{2/3} is not Lipschitz at y=0.

## 6. Exercise: The SIR Model as a System of First-Order ODEs

The **SIR model** (modèle SIR) divides a population into:
- $S(t)$: Susceptible (susceptibles)
- $I(t)$: Infected (infectés)
- $R(t)$: Recovered (guéris)

The system of first-order ODEs is:
$$\frac{dS}{dt} = -\beta S I, \quad \frac{dI}{dt} = \beta S I - \gamma I, \quad \frac{dR}{dt} = \gamma I$$

where $\beta$ is the transmission rate and $\gamma$ the recovery rate. Note $S + I + R = N$ (constant).

The **basic reproduction number** $R_0 = \beta N / \gamma$ determines whether an epidemic occurs.

**Tasks:**
1. Implement the SIR model using `solve_ivp`.
2. Plot $S(t)$, $I(t)$, $R(t)$ for parameters $\beta = 0.3$, $\gamma = 0.1$, $N = 1000$, $I(0) = 1$.
3. Vary $\beta$ and observe how $R_0$ affects the epidemic peak.
4. Verify numerically that $S(t) + I(t) + R(t) = N$ for all $t$ (conservation law).

In [ ]:
# SIR Model implementation — try it yourself first, then check below!

def sir_model(t, y, beta, gamma):
    S, I, R = y
    dSdt = -beta * S * I
    dIdt = beta * S * I - gamma * I
    dRdt = gamma * I
    return [dSdt, dIdt, dRdt]

# Parameters
N = 1000
beta, gamma = 0.3 / N, 0.1  # normalise beta by N
I0, R0_init = 1, 0
S0 = N - I0 - R0_init
R0_number = beta * N / gamma
print(f"Basic reproduction number R_0 = {R0_number:.2f}")

t_sir = (0, 160)
t_eval_sir = np.linspace(*t_sir, 500)
sol_sir = solve_ivp(sir_model, t_sir, [S0, I0, R0_init],
                    args=(beta, gamma), t_eval=t_eval_sir)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(sol_sir.t, sol_sir.y[0], 'b-', label='S (Susceptible)', linewidth=2)
axes[0].plot(sol_sir.t, sol_sir.y[1], 'r-', label='I (Infected)', linewidth=2)
axes[0].plot(sol_sir.t, sol_sir.y[2], 'g-', label='R (Recovered)', linewidth=2)
axes[0].set_xlabel('Time (days)'); axes[0].set_ylabel('Population')
axes[0].set_title(f'SIR Model ($R_0 = {R0_number:.1f}$)')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Verify conservation: S + I + R = N
total = sol_sir.y[0] + sol_sir.y[1] + sol_sir.y[2]
axes[1].plot(sol_sir.t, total - N, 'k-', linewidth=1)
axes[1].set_xlabel('Time (days)'); axes[1].set_ylabel('$S+I+R - N$')
axes[1].set_title('Conservation check (should be $\\approx 0$)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
# Figure: Left — SIR dynamics showing epidemic curve.
# Right — Numerical verification that S+I+R is conserved.